Object Oriented Porgramming (OOP) groups related data (attributes) and behaviour (methods) into reusable blueprint called classes


in FastAPI, and SQLAlchemy , every database table, Pydantic request schema, and service layers is built as a class.

## Classes , Objects and __init__

- A class is the blueprint (PascalCase naming)

- An object (or instance) is a concrete item created from that blueprint.

- __init__ is the constructor method Python runs automatically when creating a new object to initialize its state

- self refers to the specific object instance being created or called.

In [1]:
class APIService:
    def __init__(self, name: str, port: int, is_active: bool = True) -> None:
        self.name = name
        self.port = port
        self.is_acitve = is_active


# Creating two independent objects from the same class

auth_service = APIService("auth-api", 8000)
billing_service = APIService("billing-api", 8002, is_active=False)

print(auth_service.name, auth_service.port)
print(billing_service.name, billing_service.is_acitve)


auth-api 8000
billing-api False


## Instance Attributes vs Class attributes 

- instance attributes  (self.x) : Belong to one specific object. Changing auth_service.port does not affect billing_service.port

- Class attributes: Defined directly inside the class body outside __init__. Shared across all instance of that class (ideal for constants, default limits, or shared counters)

In [2]:
class RateLimiter:
    # Class attributes (shared by every instance)
    DEFAULT_MAX_REQUESTS: int = 100
    active_limiters: int = 0

    def __init__(self, client_ip: str) -> None:
        # Instance attributes (unique per client)
        self.client_ip = client_ip
        self.request_count = 0
        RateLimiter.active_limiters += 1

client_a = RateLimiter("10.0.0.1")
client_b = RateLimiter("10.0.0.2")

print(RateLimiter.active_limiters)     # Output: 2
print(client_a.DEFAULT_MAX_REQUESTS)   # Output: 100
print(client_b.DEFAULT_MAX_REQUESTS)

2
100
100


## The three method types

1) Instance Method:

- Decorator = None
- First Parameter = self
- Accesses = Instnace + Class data
- Common Backend Use = Reading/updating a specific record state

2) Class Method:

- Decorator = @classmethod
- First Parameter = cls
- Accesses = Class data only
- Common Backend Use = ALternaive constructors (eg - from_dict, from_env)

3) Static Method:
- Decorator = @staticmethod
- First Parameter = None
- Accesses = Neither self or cls
- Common Backend Use = Pure utility/validation helpers logic grouped with the class



In [3]:
class UserAccount:
    MIN_PASSWORD_LEN: int = 8

    def __init__(self, username: str, email: str, role: str = "viewer") -> None:
        self.username = username
        self.email = email
        self.role = role
        self.is_verified = False

    # 1. Instance Method: operates on a specific user object
    def verify_email(self) -> str:
        self.is_verified = True
        return f"User '{self.username}' is now verified."

    # 2. Class Method: alternative constructor from an API JSON dictionary
    @classmethod
    def from_payload(cls, payload: dict[str, str]) -> "UserAccount":
        return cls(
            username=payload["username"].strip(),
            email=payload["email"].strip().lower(),
            role=payload.get("role", "viewer"),
        )

    # 3. Static Method: standalone validation helper tied to UserAccount
    @staticmethod
    def is_strong_password(password: str) -> bool:
        return len(password) >= UserAccount.MIN_PASSWORD_LEN and any(c.isdigit() for c in password)

# Using all three method types:
raw_json = {"username": "devanath", "email": "Devanath@Company.com", "role": "intern"}
user = UserAccount.from_payload(raw_json)          # Calls @classmethod
print(user.verify_email())                         # Calls instance method
print(UserAccount.is_strong_password("pass1234"))  # Calls @staticmethod -> True

User 'devanath' is now verified.
True


classmethods often parses or returns into an Object

In [6]:
string = "2026-11-20T10:00:00 | ERROR | /api/v1/orders | 420.5"

ls = [s.strip() for s in string.split("|")]
print(ls)

['2026-11-20T10:00:00', 'ERROR', '/api/v1/orders', '420.5']


In [7]:
class LogEntry:
    VALID_LEVELS = ("INFO", "WARNING","ERROR")

    def __init__(self, timestamp, level, endpoint, latency_ms):
        self.timestamp = timestamp
        self.level = level.upper()
        self.endpoint = endpoint
        self.latency_ms = latency_ms


    def is_slow(self, threshold_ms: float = 250.0) -> bool:
        return self.latency_ms > threshold_ms

    @classmethod
    def from_raw_line(cls, line: str) -> "LogEntry":

        ls = [l.strip() for l in line.split("|")]
        timestamp, level, endpoint, raw_latency = ls

        return cls(
            timestamp=timestamp,
            level=level,
            endpoint=endpoint,
            latency_ms=float(raw_latency),
        )

    @staticmethod
    def is_valid_level(level: str) -> bool:
        return level.strip().upper() in LogEntry.VALID_LEVELS


raw = "2026-11-20T10:00:00 | ERROR | /api/v1/orders | 420.5"

entry = LogEntry.from_raw_line(raw)

print(f"Endpoint : {entry.endpoint} | Latency : {entry.latency_ms} ms")
print("Is slow (>250ms)? ", entry.is_slow())
print("Valid level 'ERROR' ? ", LogEntry.is_valid_level(entry.level))
print("Valid level 'DEBUG' ?", LogEntry.is_valid_level("DEBUG"))



Endpoint : /api/v1/orders | Latency : 420.5 ms
Is slow (>250ms)?  True
Valid level 'ERROR' ?  True
Valid level 'DEBUG' ? False
